# DeepGCN with Residual Connections on OGBN-Proteins

Node Classification on ogbn-proteins: Very deep GNNs (up to 28+ layers) with residual and dense skip connections. This notebook implements the approach with `DeepGCNLayer` inside a `K3DeepGCN` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `DeepGCNLayer` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import models as k3_models

title = "DeepGCN with Residual Connections on Proteins"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. DeepGCN Architecture using GENConv
class K3DeepGCN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4):
        super().__init__()
        self.node_encoder = layers.Dense(hidden_channels)
        # `Model.layers` is a reserved read-only property in Keras 3;
        # use a differently-named attribute for the DeepGCNLayer stack.
        self.gnn_layers = []
        for _ in range(num_layers):
            conv = k3_layers.GENConv(hidden_channels, hidden_channels, aggr="softmax")
            norm = layers.LayerNormalization()
            act = layers.Activation("relu")
            layer = k3_models.DeepGCNLayer(conv, norm, act, block="res+")
            self.gnn_layers.append(layer)
        self.lin = layers.Dense(out_channels)

    def call(self, x, edge_index):
        x = self.node_encoder(x)
        for layer in self.gnn_layers:
            x = layer(x, edge_index)
        return self.lin(x)

k3_model = K3DeepGCN(in_channels=8, hidden_channels=32, out_channels=112, num_layers=3)

# 2. Forward pass test
num_nodes = 60
dummy_x = keras.random.normal((num_nodes, 8))
dummy_edges = ops.convert_to_tensor([[0, 1], [1, 0]], dtype="int64")

out = k3_model(dummy_x, dummy_edges)
print(f"DeepGCN forward pass successful! Output shape: {out.shape}")

print("\n✓ K3-Node DeepGCN execution completed successfully!")